In [1]:
import utils
import numpy as np
import pandas as pd
import os
import importlib
import re
from matplotlib.pyplot import cm

importlib.reload(utils)

import matplotlib.pyplot as plt
import seaborn as sns # Loads in the theme

pd.set_option('future.no_silent_downcasting', True)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', 20)
#pd.set_option('display.width', 1000)

In [2]:
# Make output folder
output_dir = os.path.join('..','..','plots','behavioural', 'joystick_hesitations')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [3]:
# Preparation
pilot_analysis = False
plot_range = [0, 6]

if pilot_analysis:
    raw_output_path = '/Volumes/project/3025011.02/pilot_data/phd_1_task/experiment_output/behavioural_files_pilot_2'
    unique_subject_ids = [802,803,804,805,806,808,809,811,813,814,815]
    unique_sessions = [1]
else:
    raw_output_path = '/Volumes/project/3025011.02/raw/'
    unique_subject_ids = [5, 10, 11, 14]
    unique_sessions = [2, 3, 4]
recent_results = utils.analysis_functions.list_files_with_date_and_subject_id(raw_output_path, unique_subject_ids, unique_sessions)

In [4]:
if not recent_results:
    raise Exception('No files found')
print(recent_results)

['/Volumes/project/3025011.02/raw/sub-005/ses-mri02/beh/behavioural_output_sub-005_session-02_skyra_2025-03-14_11-00-23.csv', '/Volumes/project/3025011.02/raw/sub-005/ses-mri03/beh/behavioural_output_sub-005_session-03_skyra_2025-03-21_15-48-02.csv', '/Volumes/project/3025011.02/raw/sub-005/ses-mri04/beh/behavioural_output_sub-005_session-04_skyra_2025-03-28_15-50-17.csv', '/Volumes/project/3025011.02/raw/sub-010/ses-mri02/beh/behavioural_output_sub-010_session-02_skyra_2025-03-13_11-21-05.csv', '/Volumes/project/3025011.02/raw/sub-010/ses-mri03/beh/behavioural_output_sub-010_session-03_skyra_2025-03-19_11-41-24.csv', '/Volumes/project/3025011.02/raw/sub-010/ses-mri04/beh/behavioural_output_sub-010_session-04_skyra_2025-03-25_15-18-37.csv', '/Volumes/project/3025011.02/raw/sub-014/ses-mri02/beh/behavioural_output_sub-014_session-02_skyra_2025-04-01_16-44-40.csv', '/Volumes/project/3025011.02/raw/sub-014/ses-mri03/beh/behavioural_output_sub-014_session-03_skyra_2025-04-08_10-36-56.csv',

### Remove invalid files

In [5]:
# Remove files that contain invalid data
raw_output_path = '/Volumes/project/3025011.02/raw/'
invalid_data = pd.read_csv(f'{raw_output_path}../incomplete_data.csv', delimiter=';')
combinations = {
    f"sub-{row['subject-id']:03}_session-{row['session-number']:02}"
    for _, row in invalid_data[invalid_data['datatype'] == 'beh'].iterrows()
}
print(f'These participants will be excluded: {combinations}\n')

patterns = [re.compile(re.escape(combo)) for combo in combinations]
# Filter the list: remove any string that contains one of the combinations
recent_results = [
    item for item in recent_results
    if not any(pattern.search(item) for pattern in patterns)
]
print(recent_results)

These participants will be excluded: {'sub-003_session-03'}
['/Volumes/project/3025011.02/raw/sub-005/ses-mri02/beh/behavioural_output_sub-005_session-02_skyra_2025-03-14_11-00-23.csv', '/Volumes/project/3025011.02/raw/sub-005/ses-mri03/beh/behavioural_output_sub-005_session-03_skyra_2025-03-21_15-48-02.csv', '/Volumes/project/3025011.02/raw/sub-005/ses-mri04/beh/behavioural_output_sub-005_session-04_skyra_2025-03-28_15-50-17.csv', '/Volumes/project/3025011.02/raw/sub-010/ses-mri02/beh/behavioural_output_sub-010_session-02_skyra_2025-03-13_11-21-05.csv', '/Volumes/project/3025011.02/raw/sub-010/ses-mri03/beh/behavioural_output_sub-010_session-03_skyra_2025-03-19_11-41-24.csv', '/Volumes/project/3025011.02/raw/sub-010/ses-mri04/beh/behavioural_output_sub-010_session-04_skyra_2025-03-25_15-18-37.csv', '/Volumes/project/3025011.02/raw/sub-014/ses-mri02/beh/behavioural_output_sub-014_session-02_skyra_2025-04-01_16-44-40.csv', '/Volumes/project/3025011.02/raw/sub-014/ses-mri03/beh/behaviour

In [6]:
combined_results_dataframe = utils.analysis_functions.combine_result_files(recent_results)
if pilot_analysis:
    combined_results_dataframe.rename(columns={'iti': 'trial_duration'}, inplace=True)
    combined_results_dataframe['stimuli_type'] = combined_results_dataframe['stimuli_type'].replace(3, 2)
print(combined_results_dataframe)

      trial  emotional_cue_time response objectively_correct  \
0         0        1.741947e+09     down                Even   
1         1        1.741947e+09       up                Even   
2         2        1.741947e+09       up                Even   
3         3        1.741947e+09     down                Even   
4         4        1.741947e+09     down                Even   
5         5        1.741947e+09     down                Even   
6         6        1.741947e+09       up                Even   
7         7        1.741947e+09     down                Even   
8         8        1.741947e+09       up               False   
9         9        1.741947e+09       up                True   
10       10        1.741947e+09     down               False   
11       11        1.741947e+09       up               False   
12       12        1.741947e+09       up               False   
13       13        1.741947e+09       up                True   
14       14        1.741947e+09     down

## Flip results if joystick orientation was incorrect
Specifically, switch the response and probability

In [7]:
# List of subject_session combinations to update
match_list = ['sub-004_session-04']

# Create a new column that combines subject_id and session in the same format as the list.
combined_results_dataframe['sub_session'] = combined_results_dataframe['subject_id'] + '_' + combined_results_dataframe['session']

# Create a mask for rows that match any of the combinations in match_list.
mask = combined_results_dataframe['sub_session'].isin(match_list)

# For 'probability_condition', swap 80 with 20 and vice versa.
combined_results_dataframe.loc[mask, 'probability_condition'] = (
    combined_results_dataframe.loc[mask, 'probability_condition']
    .replace({80: 20, 20: 80})
)

# For 'response', swap 'up' with 'down' and vice versa.
combined_results_dataframe.loc[mask, 'response'] = (
    combined_results_dataframe.loc[mask, 'response']
    .replace({'up': 'down', 'down': 'up'})
)

# Optionally, drop the helper column
combined_results_dataframe.drop(columns='sub_session', inplace=True)

print(combined_results_dataframe)

      trial  emotional_cue_time response objectively_correct  \
0         0        1.741947e+09     down                Even   
1         1        1.741947e+09       up                Even   
2         2        1.741947e+09       up                Even   
3         3        1.741947e+09     down                Even   
4         4        1.741947e+09     down                Even   
5         5        1.741947e+09     down                Even   
6         6        1.741947e+09       up                Even   
7         7        1.741947e+09     down                Even   
8         8        1.741947e+09       up               False   
9         9        1.741947e+09       up                True   
10       10        1.741947e+09     down               False   
11       11        1.741947e+09       up               False   
12       12        1.741947e+09       up               False   
13       13        1.741947e+09       up                True   
14       14        1.741947e+09     down

# Analysis
We'll investigate the joystick hesitations in the task.

In [8]:
# Count amount of subplots necessary based on subject_id's and sessions
## Finds unique values
unique_subject_ids = combined_results_dataframe['subject_id'].unique()
unique_sessions = combined_results_dataframe['session'].unique()

## Calculates the number of subplots needed
n_rows = len(unique_subject_ids)
n_cols = len(unique_sessions)
n_subplots = len(unique_sessions) * len(unique_subject_ids)

### Make additional columns for congruency, valence, WSLS and volatility

In [9]:
# Add congruency column
combined_results_dataframe['congruency'] = combined_results_dataframe.apply(utils.analysis_functions.check_congruency, axis=1)
print(combined_results_dataframe[['stimuli_type','probability_condition','congruency']])

      stimuli_type  probability_condition   congruency
0                1                     50    undefined
1                1                     50    undefined
2                1                     50    undefined
3                2                     50    undefined
4                2                     50    undefined
5                2                     50    undefined
6                2                     50    undefined
7                1                     50    undefined
8                1                     20  incongruent
9                2                     80  incongruent
10               2                     80  incongruent
11               1                     20  incongruent
12               1                     20  incongruent
13               2                     80  incongruent
14               1                     20  incongruent
15               1                     20  incongruent
16               2                     80  incongruent
17        

In [10]:
# Add valence column
combined_results_dataframe['valence'] = combined_results_dataframe.apply(utils.analysis_functions.check_valence, axis=1)
print(combined_results_dataframe[['stimuli_type','valence']])

      stimuli_type valence
0                1   angry
1                1   angry
2                1   angry
3                2   happy
4                2   happy
5                2   happy
6                2   happy
7                1   angry
8                1   angry
9                2   happy
10               2   happy
11               1   angry
12               1   angry
13               2   happy
14               1   angry
15               1   angry
16               2   happy
17               2   happy
18               1   angry
19               1   angry
20               2   happy
21               2   happy
22               2   happy
23               1   angry
24               1   angry
25               2   happy
26               1   angry
27               1   angry
28               2   happy
29               2   happy
30               2   happy
31               2   happy
32               1   angry
33               1   angry
34               2   happy
35               1   angry
3

In [11]:
# Turns the objective values into boolean ones
mapping = {'True': 1, 'False': 0, 'Even': 0, 'Late': 0}
#pd.options.display.max_rows = None

combined_results_dataframe['objectively_correct_boolean'] = combined_results_dataframe['objectively_correct'].replace(mapping)
combined_results_dataframe['objectively_correct_boolean'] = combined_results_dataframe['objectively_correct_boolean'].astype('int')

In [12]:
# Add volatility column
combined_results_dataframe = combined_results_dataframe[combined_results_dataframe['probability_condition'] != 50]
combined_results_dataframe = combined_results_dataframe.groupby('stimuli_type', group_keys=False).apply(lambda x: utils.analysis_functions.check_volatility(x))
combined_results_dataframe = combined_results_dataframe.sort_values(by=['subject_id','session','trial'], ascending=[True, True, True])
print(combined_results_dataframe)

/var/folders/bh/ld02p1_550d75y0x32dmpb540000gp/T/ipykernel_6229/1233922260.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  combined_results_dataframe = combined_results_dataframe.groupby('stimuli_type', group_keys=False).apply(lambda x: utils.analysis_functions.check_volatility(x))


      trial  emotional_cue_time response objectively_correct  \
0         8        1.741947e+09       up               False   
0         9        1.741947e+09       up                True   
1        10        1.741947e+09     down               False   
1        11        1.741947e+09       up               False   
2        12        1.741947e+09       up               False   
2        13        1.741947e+09       up                True   
3        14        1.741947e+09     down                True   
4        15        1.741947e+09     down                True   
3        16        1.741947e+09       up                True   
4        17        1.741947e+09       up                True   
5        18        1.741947e+09     down                True   
6        19        1.741947e+09       up               False   
5        20        1.741947e+09     down               False   
6        21        1.741947e+09       up                True   
7        22        1.741947e+09       up

# Now, we have to import the raw joystick data and merge it with the combined results dataframe

In [13]:
recent_results = utils.analysis_functions.list_files_with_date_and_subject_id(raw_output_path, subject_id_list, sessions_list, 'joystick_output')

sub-014 did not complete session-03


In [14]:
joystick_dataframe = utils.analysis_functions.combine_joystick_with_results(combined_results_dataframe, recent_results)

     joystick_datapoint  trial  joystick_location  joystick_timepoint  \
1                     1      0                607        1.741947e+09   
2                     2      0                607        1.741947e+09   
3                     3      0                607        1.741947e+09   
4                     4      0                607        1.741947e+09   
5                     5      0                607        1.741947e+09   
6                     6      0                607        1.741947e+09   
7                     7      0                607        1.741947e+09   
8                     8      0                607        1.741947e+09   
9                     9      0                607        1.741947e+09   
10                   10      0                607        1.741947e+09   
11                   11      0                607        1.741947e+09   
12                   12      0                607        1.741947e+09   
13                   13      0                607  

In [24]:
print(joystick_dataframe[(joystick_dataframe['trial'] == 15) & (joystick_dataframe['subject_id'] == 'sub-005') & (joystick_dataframe['session'] == 'session-02')])

     trial  emotional_cue_time response objectively_correct  \
636     15        1.741947e+09     down                True   
637     15        1.741947e+09     down                True   
638     15        1.741947e+09     down                True   
639     15        1.741947e+09     down                True   
640     15        1.741947e+09     down                True   
641     15        1.741947e+09     down                True   
642     15        1.741947e+09     down                True   
643     15        1.741947e+09     down                True   
644     15        1.741947e+09     down                True   
645     15        1.741947e+09     down                True   
646     15        1.741947e+09     down                True   
647     15        1.741947e+09     down                True   
648     15        1.741947e+09     down                True   
649     15        1.741947e+09     down                True   
650     15        1.741947e+09     down                